In [1]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd


# -------------------------
# 1. Check probability validity
# -------------------------
def s_nan(y_prob, eps=1e-3):
    if not np.all(np.isfinite(y_prob)):
        return 0

    row_sums = np.sum(y_prob, axis=1)
    if np.all(np.abs(row_sums - 1.0) <= eps):
        return 1
    return 0


# -------------------------
# 2. Majority + JSD score
# -------------------------
def s_maj_jsd(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_true))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    # Majority score
    f_max = np.max(pred_dist)
    s_maj = 1 - (f_max - 1/n_classes) / (1 - 1/n_classes)

    # Jensen-Shannon divergence
    # jensenshannon() trả về căn bậc hai của JSD dùng log e
    js_distance = jensenshannon(true_dist, pred_dist)
    
    # Bình phương để lấy JSD (Divergence)
    js_divergence = js_distance**2
    s_jsd = 1 - (js_divergence / np.log(2))

    return s_maj, s_jsd


# -------------------------
# 3. Entropy score
# -------------------------
def s_ent(y_prob):
    N, K = y_prob.shape

    h = -np.sum(y_prob * np.log(y_prob + 1e-10), axis=1) / np.log(K)
    m = np.median(h)

    return 1 - min(abs(m - 0.5) / 0.5, 1.0)


# -------------------------
# 4. Drift score
# -------------------------
def s_drift(y_pred, ref_dist=None, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_pred))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)

    if ref_dist is None:
        ref_dist = np.ones(n_classes) / n_classes

    # jensenshannon trả về căn bậc hai của JSD (Distance)
    js_distance = jensenshannon(pred_dist, ref_dist)
    
    # Bình phương để có JSD (Divergence)
    js_divergence = js_distance**2
    
    return 1 - (js_divergence / np.log(2))


# -------------------------
# 5. Efficiency score
# -------------------------
def s_eff(mask):
    return np.mean(mask)


# -------------------------
# 6. Leakage score (generic probe)
# -------------------------
def s_leak_1(mask, y_true, probe_model, test_size=0.3, random_state=40):

    X_train, X_test, y_train, y_test = train_test_split(
        mask, y_true,
        test_size=test_size,
        random_state=random_state,
        stratify=y_true
    )

    probe_model.fit(X_train, y_train)

    if hasattr(probe_model, "predict_proba"):
        y_score = probe_model.predict_proba(X_test)
    else:
        y_score = probe_model.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score, multi_class="ovr")
    return 1 - auc


# -------------------------
# 7. Logistic leakage
# -------------------------
def s_leak(X, y, test_size=0.3, random_state=40):

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    probe = LogisticRegression(max_iter=1000)
    probe.fit(Xtr, ytr)

    y_score = probe.predict_proba(Xte)

    auc = roc_auc_score(yte, y_score, multi_class="ovr")
    return float(1 - auc)

# -------------------------
# 8. Final geometric mean score
# -------------------------
def S_san_plus(values):
    values = np.clip(values, 1e-10, 1)
    return np.prod(values) ** (1 / len(values))


In [2]:
def compute_ref_dist(y_train):
    y = y_train.astype(int).to_numpy()
    n_classes = len(np.unique(y))
    return np.bincount(y, minlength=n_classes) / len(y)

def run_sanity_evaluation(
    *,
    path_1,
    path_2,
    version_name,
    train_name,
    n_classes=3
):
    print(
        f"[START] Sanity evaluation | "
        f"train={train_name} | version={version_name}"
    )

    rows = []

    # -----------------
    # Load train
    # -----------------
    train_df = pd.read_csv(f"{path_1}/{train_name}.csv")
    y_train = train_df["label_3"]
    ref_dist = compute_ref_dist(y_train)

    # -----------------
    # Loop phases
    # -----------------
    for phase in [1, 2, 3, 4]:
        print(f"  -> Running phase {phase}...")

        # ---- probability matrix
        prob_df = pd.read_csv(
            f"{path_2}/probability_matrix_{version_name}_phase{phase}.csv"
        )

        y_true = prob_df["y_true"].astype(int).to_numpy()
        y_pred = prob_df["y_pred"].astype(int).to_numpy()
        y_prob = prob_df[
            ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
        ].to_numpy()

        # ---- test data (for leakage & efficiency)
        test_df = pd.read_csv(f"{path_1}/test_{phase}.csv")
        X_test = test_df.drop(columns=["label_3"], errors="ignore")
        mask = X_test.notna().astype(int).values

        # ---- metrics
        smaj, sjsd = s_maj_jsd(y_true, y_pred, n_classes=n_classes)

        snan   = s_nan(y_prob)
        sent   = s_ent(y_prob)
        sdrift = s_drift(y_pred, ref_dist, n_classes=n_classes)
        sleak  = s_leak(mask, y_true)
        s_eff_ = s_eff(mask)

        s_san_plus = S_san_plus([
            snan,
            sjsd,
            sent,
            sdrift,
            s_eff_,
            sleak
        ])

        rows.append({
            "train_name": train_name,
            "version_name": version_name,
            "phase": phase,
            "snan": snan,
            "smaj": smaj,
            "sjsd": sjsd,
            "sent": sent,
            "sdrift": sdrift,
            "sleak": sleak,
            "s_san+": s_san_plus,
            "s_eff": s_eff_
        })

    print(
        f"[END] Sanity evaluation DONE | "
        f"rows={len(rows)} | "
        f"phases=4"
    )

    columns_order = [
        "train_name",
        "version_name",
        "phase",
        "snan",
        "smaj",
        "sjsd",
        "sent",
        "sdrift",
        "sleak",
        "s_san+",
        "s_eff"
    ]

    return pd.DataFrame(rows)[columns_order]


# Chạy với các version data

## V1 (Median)

In [3]:
path_1 = "/kaggle/input/lo-dataset/Median/Median"

In [4]:
path_2 = "/kaggle/input/datasets/anhtran10/bilstm-results/BiLSTM_results"

In [5]:
df_v1 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V1 (Median)",
    train_name="train_median"
)
df_v1

[START] Sanity evaluation | train=train_median | version=V1 (Median)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median,V1 (Median),1,1,0.005117,0.999703,0.001353,0.999704,0.5,0.296264,1.0
1,train_median,V1 (Median),2,1,0.059935,0.985470,0.015046,0.985470,0.5,0.440499,1.0
2,train_median,V1 (Median),3,1,0.007059,0.999807,0.000810,0.999807,0.5,0.271974,1.0
3,train_median,V1 (Median),4,1,0.004782,0.999988,0.000004,0.999988,0.5,0.114038,1.0


In [6]:
df_v1.to_csv("results_v1.csv", index=False)

## V2 (Median CDSMOTE)

In [7]:
df_v2 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V2 (Median CDSMOTE)",
    train_name="train_median_cdsmote"
)
df_v2

[START] Sanity evaluation | train=train_median_cdsmote | version=V2 (Median CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_cdsmote,V2 (Median CDSMOTE),1,1,0.001149,0.998971,5.252988e-08,0.544843,0.5,0.049264,1.0
1,train_median_cdsmote,V2 (Median CDSMOTE),2,1,0.001123,0.999063,3.074823e-08,0.544835,0.5,0.045058,1.0
2,train_median_cdsmote,V2 (Median CDSMOTE),3,1,0.002568,0.999702,4.546182e-09,0.549093,0.5,0.032811,1.0
3,train_median_cdsmote,V2 (Median CDSMOTE),4,1,0.008673,0.999800,3.828206e-06,0.563341,0.5,0.101261,1.0


In [8]:
df_v2.to_csv("results_v2.csv", index=False)

## V3 (Median SASMOTE)

In [9]:
df_v3 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V3 (Median SASMOTE)",
    train_name="train_median_sasmote"
)
df_v3

[START] Sanity evaluation | train=train_median_sasmote | version=V3 (Median SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_sasmote,V3 (Median SASMOTE),1,1,0.003188,0.999753,0.000004,0.550758,0.5,0.100742,1.0
1,train_median_sasmote,V3 (Median SASMOTE),2,1,0.002555,0.999738,0.000002,0.548791,0.5,0.093877,1.0
2,train_median_sasmote,V3 (Median SASMOTE),3,1,0.012674,0.999053,0.000339,0.569551,0.5,0.214136,1.0
3,train_median_sasmote,V3 (Median SASMOTE),4,1,0.009015,0.999769,0.000005,0.563976,0.5,0.106241,1.0


In [10]:
df_v3.to_csv("results_v3.csv", index=False)

## V4 (Median RadiusSMOTE)

In [11]:
df_v4 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V4 (Median RadiusSMOTE)",
    train_name="train_median_radiussmote"
)
df_v4

[START] Sanity evaluation | train=train_median_radiussmote | version=V4 (Median RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_radiussmote,V4 (Median RadiusSMOTE),1,1,0.001265,0.999263,0.000007,0.545046,0.5,0.112557,1.0
1,train_median_radiussmote,V4 (Median RadiusSMOTE),2,1,0.003317,0.999879,0.000030,0.550987,0.5,0.142247,1.0
2,train_median_radiussmote,V4 (Median RadiusSMOTE),3,1,0.005027,0.999997,0.000097,0.555084,0.5,0.173267,1.0
3,train_median_radiussmote,V4 (Median RadiusSMOTE),4,1,0.007008,0.999943,0.000098,0.559700,0.5,0.173571,1.0


In [12]:
df_v4.to_csv("results_v4.csv", index=False)

## V5 (Mean)

In [13]:
path_1 = "/kaggle/input/lo-dataset/Mean/Mean"

In [14]:
df_v5 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V5 (Mean)",
    train_name="train_mean"
)
df_v5

[START] Sanity evaluation | train=train_mean | version=V5 (Mean)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean,V5 (Mean),1,1,0.001600,0.999436,0.000053,0.999436,0.5,0.172468,1.0
1,train_mean,V5 (Mean),2,1,0.002839,0.999812,0.000017,0.999812,0.5,0.142485,1.0
2,train_mean,V5 (Mean),3,1,0.004601,0.999966,0.000117,0.999966,0.5,0.197046,1.0
3,train_mean,V5 (Mean),4,1,0.004898,0.999994,0.000009,0.999994,0.5,0.129426,1.0


In [15]:
df_v5.to_csv("results_v5.csv", index=False)

## V6 (Mean CDSMOTE)

In [16]:
df_v6 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V6 (Mean CDSMOTE)",
    train_name="train_mean_cdsmote"
)
df_v6

[START] Sanity evaluation | train=train_mean_cdsmote | version=V6 (Mean CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_cdsmote,V6 (Mean CDSMOTE),1,1,0.001955,0.999550,0.000401,0.547060,0.5,0.218741,1.0
1,train_mean_cdsmote,V6 (Mean CDSMOTE),2,1,0.002904,0.999793,0.000121,0.549973,0.5,0.179357,1.0
2,train_mean_cdsmote,V6 (Mean CDSMOTE),3,1,0.003259,0.999820,0.000099,0.550930,0.5,0.173379,1.0
3,train_mean_cdsmote,V6 (Mean CDSMOTE),4,1,0.007376,0.999920,0.000111,0.560471,0.5,0.177354,1.0


In [17]:
df_v6.to_csv("results_v6.csv", index=False)

## V7 (Mean SASMOTE)

In [18]:
df_v7 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V7 (Mean SASMOTE)",
    train_name="train_mean_sasmote"
)
df_v7

[START] Sanity evaluation | train=train_mean_sasmote | version=V7 (Mean SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_sasmote,V7 (Mean SASMOTE),1,1,0.003485,0.998510,4.882507e-10,0.550396,0.5,0.022626,1.0
1,train_mean_sasmote,V7 (Mean SASMOTE),2,1,0.001007,0.998838,8.545117e-09,0.544341,0.5,0.036392,1.0
2,train_mean_sasmote,V7 (Mean SASMOTE),3,1,0.001800,0.999483,8.209595e-09,0.546941,0.5,0.036183,1.0
3,train_mean_sasmote,V7 (Mean SASMOTE),4,1,0.010209,0.999625,1.753688e-05,0.566395,0.5,0.130611,1.0


In [19]:
df_v7.to_csv("results_v7.csv", index=False)

## V8 (Mean RadiusSMOTE)

In [20]:
df_v8 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V8 (Mean RadiusSMOTE)",
    train_name="train_mean_radiussmote"
)
df_v8

[START] Sanity evaluation | train=train_mean_radiussmote | version=V8 (Mean RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_radiussmote,V8 (Mean RadiusSMOTE),1,1,0.004465,0.999846,0.000223,0.554006,0.5,0.198863,1.0
1,train_mean_radiussmote,V8 (Mean RadiusSMOTE),2,1,0.004749,0.999777,0.000117,0.554686,0.5,0.178714,1.0
2,train_mean_radiussmote,V8 (Mean RadiusSMOTE),3,1,0.014771,0.998346,0.000478,0.575605,0.5,0.227126,1.0
3,train_mean_radiussmote,V8 (Mean RadiusSMOTE),4,1,0.007763,0.999891,0.000035,0.561203,0.5,0.146671,1.0


In [21]:
df_v8.to_csv("results_v8.csv", index=False)

## V9 (Extra Trees)

In [22]:
path_1 = "/kaggle/input/lo-dataset/Extra_trees/Extra_trees"

In [23]:
df_v9 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V9 (MissForest)",
    train_name="train_extra"
)
df_v9

[START] Sanity evaluation | train=train_extra | version=V9 (MissForest)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra,V9 (MissForest),1,1,0.001458,0.999333,6.894147e-06,0.999334,0.5,0.122879,1.0
1,train_extra,V9 (MissForest),2,1,0.002659,0.999700,8.023205e-07,0.999700,0.5,0.085870,1.0
2,train_extra,V9 (MissForest),3,1,0.006395,0.999934,6.106942e-07,0.999934,0.5,0.082058,1.0
3,train_extra,V9 (MissForest),4,1,0.004853,0.999994,5.232999e-08,0.999994,0.5,0.054486,1.0


In [24]:
df_v9.to_csv("results_v9.csv", index=False)

## V10 (Extra Trees CDSMOTE)

In [25]:
df_v10 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V10 (MissForest CDSMOTE)",
    train_name="train_extra_cdsmote"
)
df_v10

[START] Sanity evaluation | train=train_extra_cdsmote | version=V10 (MissForest CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_cdsmote,V10 (MissForest CDSMOTE),1,1,0.000077,0.998338,0.0,0.541205,0.5,0.017322,1.0
1,train_extra_cdsmote,V10 (MissForest CDSMOTE),2,1,0.000232,0.998493,0.0,0.541813,0.5,0.017326,1.0
2,train_extra_cdsmote,V10 (MissForest CDSMOTE),3,1,0.003665,0.999659,0.0,0.551947,0.5,0.017383,1.0
3,train_extra_cdsmote,V10 (MissForest CDSMOTE),4,1,0.007731,0.999885,0.0,0.561344,0.5,0.017432,1.0


In [26]:
df_v10.to_csv("results_v10.csv", index=False)

## V11 (Extra Trees SASMOTE)

In [27]:
df_v11 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V11 (MissForest SASMOTE)",
    train_name="train_extra_sasmote"
)
df_v11

[START] Sanity evaluation | train=train_extra_sasmote | version=V11 (MissForest SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_sasmote,V11 (MissForest SASMOTE),1,1,0.000123,0.998383,0.0,0.541384,0.5,0.017323,1.0
1,train_extra_sasmote,V11 (MissForest SASMOTE),2,1,0.000413,0.998662,0.0,0.542490,0.5,0.017330,1.0
2,train_extra_sasmote,V11 (MissForest SASMOTE),3,1,0.004272,0.999957,0.0,0.553139,0.5,0.017390,1.0
3,train_extra_sasmote,V11 (MissForest SASMOTE),4,1,0.007408,0.999903,0.0,0.560682,0.5,0.017429,1.0


In [28]:
df_v11.to_csv("results_v11.csv", index=False)

## V12 (Extra Trees RadiusSMOTE)

In [29]:
df_v12 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V12 (MissForest RadiusSMOTE)",
    train_name="train_extra_radiussmote"
)
df_v12

[START] Sanity evaluation | train=train_extra_radiussmote | version=V12 (MissForest RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),1,1,0.251702,0.920134,0.234077,0.742144,0.5,0.656314,1.0
1,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),2,1,0.509509,0.815331,0.186513,0.796277,0.5,0.626632,1.0
2,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),3,1,0.129297,0.963314,0.024684,0.680057,0.5,0.448006,1.0
3,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),4,1,0.006743,0.999956,0.000001,0.559134,0.5,0.084419,1.0


In [30]:
df_v12.to_csv("results_v12.csv", index=False)